# <font color='blue'> Optimization Algorithms for Deep Learning </font>

Gradient Descent provides the fundamental mechanism by which neural networks learn, iteratively updating the model parameters in the direction that reduces the loss function. Although conceptually simple, standard Gradient Descent often performs poorly when training modern deep neural networks. High-dimensional optimization landscapes, noisy gradient estimates, ill-conditioned curvature, and millions of trainable parameters make efficient optimization considerably more challenging than the simple examples encountered in introductory discussions.

To overcome these difficulties, a variety of optimization algorithms have been developed that improve the speed, stability, and reliability of neural network training. These methods extend the basic Gradient Descent algorithm by incorporating ideas such as momentum, adaptive learning rates, and parameter-specific update rules. Many of the optimizers used in contemporary deep learning—including RMSProp and Adam—are direct extensions of these principles.

This chapter motivates the need for advanced optimization algorithms, examines the limitations of vanilla Gradient Descent, and introduces the major families of optimizers that have become standard in modern deep learning.

---

# <font color='orange'> 1. Why Isn't Gradient Descent Enough? </font>

Gradient Descent updates the parameters using

$$
\boxed{
\mathbf{W}
\leftarrow
\mathbf{W}
-
\eta
\nabla J
}
$$

This rule is mathematically elegant,

but practical neural networks often contain

```
Millions

or

Billions

of parameters.
```

The optimization problem therefore becomes much more difficult than minimizing a simple quadratic function.

---

# <font color='orange'> 2. The Challenges of Deep Learning Optimization </font>

Modern neural networks present several optimization challenges.

* Very high-dimensional parameter spaces.
* Highly non-convex loss landscapes.
* Noisy gradients from mini-batches.
* Poorly scaled gradients across different layers.
* Computational efficiency.

These challenges motivate more sophisticated optimization algorithms.

---

# <font color='orange'> 3. Slow Convergence </font>

Consider a narrow valley in the loss landscape.

```
Loss

│\

│ \

│  \

│   \____

└────────────
```

Standard Gradient Descent often moves

```
↓

←

↓

→

↓

←
```

zigzagging toward the minimum.

This results in unnecessarily slow convergence.

---

# <font color='orange'> 4. Ravines and Ill-Conditioning </font>

Many optimization problems contain

**ravines**,

where the curvature differs greatly between directions.

Conceptually,

```
Very Steep

↓

_____________

← Slow Progress →
```

The optimizer oscillates across the steep direction

while making only gradual progress along the shallow direction.

Advanced optimizers reduce this oscillatory behaviour.

---

# <font color='orange'> 5. Noisy Mini-Batch Gradients </font>

Mini-batch training estimates the gradient using only a subset of the training data.

Consequently,

the estimated gradient is only an approximation of the true gradient.

Conceptually,

```
True Direction

↓

Estimated Direction

↓

Slightly Different
```

This noise can slow optimization,

but it can also help the optimizer escape undesirable regions such as shallow local minima or saddle points.

---

# <font color='orange'> 6. Vanishing and Exploding Updates </font>

During training,

some parameters may receive

very small updates,

while others receive

very large updates.

This imbalance can slow learning or make optimization unstable.

Adaptive optimization algorithms attempt to compensate for these differences by adjusting the learning rate for individual parameters.

---

# <font color='orange'> 7. Desired Properties of an Optimizer </font>

An effective optimizer should

* converge rapidly,
* remain numerically stable,
* handle noisy gradients,
* require minimal manual tuning,
* scale to very large neural networks.

Modern optimizers are designed with these objectives in mind.

---

# <font color='orange'> 8. Families of Optimization Algorithms </font>

Most modern optimizers belong to one of several broad categories.

| Family | Primary Idea |
| :--- | :--- |
| Gradient Descent | Fixed learning rate |
| Momentum Methods | Accumulate previous updates |
| Adaptive Methods | Adjust learning rates automatically |
| Combined Methods | Use both momentum and adaptive learning rates |

This evolution reflects decades of research into improving optimization performance.

---

# <font color='orange'> 9. The Major Optimizers </font>

The optimizers studied in this module are

| Optimizer | Main Innovation |
| :--- | :--- |
| Gradient Descent | Basic parameter updates |
| Momentum | Uses previous updates to accelerate convergence |
| Nesterov Accelerated Gradient (NAG) | Improves momentum by looking ahead |
| AdaGrad | Parameter-specific learning rates |
| RMSProp | Prevents AdaGrad's learning rate from shrinking excessively |
| Adam | Combines Momentum and RMSProp |
| AdamW | Decouples weight decay from gradient updates |

Each algorithm builds upon the limitations of its predecessors.

---

# <font color='orange'> 10. Choosing an Optimizer </font>

Different optimizers are appropriate for different situations.

General recommendations include

| Situation | Typical Optimizer |
| :--- | :--- |
| Educational examples | SGD |
| Image classification | SGD with Momentum or Adam |
| General deep learning | Adam |
| Large transformer models | AdamW |

There is no universally optimal optimizer.

Performance depends on the architecture, dataset, and learning task.

---

# <font color='orange'> 11. Optimizers in TensorFlow </font>

TensorFlow provides implementations of all major optimizers.

```python
import tensorflow as tf

optimizer = tf.keras.optimizers.SGD(
    learning_rate=0.01
)
```

```python
optimizer = tf.keras.optimizers.Adam(
    learning_rate=0.001
)
```

```python
optimizer = tf.keras.optimizers.RMSprop(
    learning_rate=0.001
)
```

Changing the optimizer usually requires only a single line of code.

---

# <font color='orange'> 12. Common Misconceptions </font>

### Adam Is Always the Best Optimizer

False.

Although Adam performs well across many problems, SGD with Momentum often achieves better final generalization for some computer vision tasks.

---

### Optimizers Eliminate the Need for Hyperparameter Tuning

False.

Learning rates, batch sizes, and regularization techniques remain important regardless of the optimizer.

---

### Optimization and Generalization Are the Same

False.

An optimizer minimizes the training loss.

Good optimization does not automatically guarantee good generalization to unseen data.

---

# <font color='purple'> 13. Conceptual Summary </font>

| Concept | Description |
| :--- | :--- |
| Gradient Descent | Fundamental optimization algorithm |
| Optimization Challenge | Minimize the loss efficiently in high-dimensional parameter spaces |
| Momentum | Accelerates learning using previous updates |
| Adaptive Optimizers | Automatically adjust parameter-specific learning rates |
| Adam | Combines momentum with adaptive learning rates |
| AdamW | Improved version of Adam with decoupled weight decay |
| Main Goal | Achieve faster, more stable, and more efficient training |

> **Key Insight:** Although Gradient Descent provides the foundation for neural network optimization, the complexity of modern deep learning models requires more sophisticated optimization algorithms. Challenges such as noisy gradients, ill-conditioned loss landscapes, and high-dimensional parameter spaces have led to the development of methods including Momentum, RMSProp, and Adam. These optimizers improve training efficiency by accelerating convergence, stabilizing updates, and adapting learning rates, making them indispensable tools in contemporary deep learning.

# <font color='blue'> Momentum Optimization </font>

Standard Gradient Descent updates the parameters using only the gradient computed at the current iteration. Although this approach eventually reduces the loss function, it often converges slowly, particularly in optimization landscapes containing narrow valleys or regions with highly uneven curvature. In such situations, Gradient Descent tends to oscillate across steep directions while making only gradual progress toward the minimum.

The **Momentum** optimization algorithm addresses this limitation by incorporating information from previous parameter updates. Rather than relying solely on the current gradient, Momentum accumulates a running average of past updates, allowing the optimization process to build speed in directions that consistently reduce the loss while damping oscillations in directions where the gradient changes rapidly.

This chapter introduces the intuition behind Momentum, derives the mathematical update equations, explains the role of the momentum coefficient, and discusses its practical implementation in modern deep learning frameworks.

---

# <font color='orange'> 1. Why Gradient Descent Can Be Slow </font>

Recall the Gradient Descent update rule,

$$
\boxed{
\mathbf{W}
\leftarrow
\mathbf{W}
-
\eta
\nabla J
}
$$

Each update depends only on

the current gradient.

Previous gradients

are immediately forgotten.

Consequently,

Gradient Descent may repeatedly change direction,

leading to slow convergence.

---

# <font color='orange'> 2. The Zigzag Problem </font>

Consider a narrow valley in the loss landscape.

```
Loss

│\
│ \
│  \
│   \
│    \____

└────────────►
```

Gradient Descent often behaves as

```
↓

←

↓

→

↓

←

↓

→
```

The optimizer oscillates across the valley

while making only slow progress toward the minimum.

This inefficient behaviour motivates the introduction of Momentum.

---

# <font color='orange'> 3. Physical Intuition </font>

Imagine pushing a ball down a hill.

Initially,

the ball moves slowly.

As it continues downhill,

it gains

**momentum**.

Conceptually,

```
Start

↓

Small Speed

↓

Faster

↓

Even Faster
```

The ball naturally continues moving in the same direction unless acted upon by an opposing force.

Momentum optimization applies the same principle to parameter updates.

---

# <font color='orange'> 4. The Central Idea </font>

Instead of updating the parameters using only

the current gradient,

Momentum combines

* the current gradient,
* previous updates.

Conceptually,

```
Previous Velocity

+

Current Gradient

↓

New Velocity

↓

Parameter Update
```

This allows the optimizer to maintain motion along useful directions.

---

# <font color='orange'> 5. Velocity </font>

Momentum introduces a new variable,

called the

**velocity**,

which stores a running average of previous gradients.

The velocity is updated according to

$$
\boxed{
\mathbf{v}
=
\beta
\mathbf{v}
-
\eta
\nabla J
}
$$

where

* \(\mathbf{v}\) is the velocity,
* \(\beta\) is the momentum coefficient,
* \(\eta\) is the learning rate.

The parameters are then updated using

$$
\boxed{
\mathbf{W}
\leftarrow
\mathbf{W}
+
\mathbf{v}
}
$$

Notice that the parameter update now depends on the accumulated velocity rather than the gradient alone.

---

# <font color='orange'> 6. The Momentum Coefficient </font>

The parameter

\(\beta\)

controls

how much of the previous velocity is retained.

Typical values are

```
0.9

or

0.99
```

---

### Small β

```
Little Memory

↓

Behaves Like Gradient Descent
```

---

### Large β

```
Long Memory

↓

Smooth Updates

↓

Faster Progress
```

A value of

```
β = 0.9
```

means that approximately 90% of the previous velocity contributes to the next update.

---

# <font color='orange'> 7. Why Momentum Works </font>

Suppose consecutive gradients point

in approximately the same direction.

Each update reinforces

the previous one.

Conceptually,

```
Gradient

↓

Gradient

↓

Gradient

↓

Large Velocity
```

The optimizer accelerates,

allowing faster movement through shallow regions of the loss landscape.

---

# <font color='orange'> 8. Reducing Oscillations </font>

Now consider

a steep valley.

The gradients frequently change sign across the steep direction.

Conceptually,

```
←

→

←

→
```

Momentum averages these opposing updates,

reducing unnecessary oscillations.

The optimizer therefore follows a smoother trajectory toward the minimum.

---

# <font color='orange'> 9. Numerical Example </font>

Suppose

* learning rate

$$
\eta=0.1,
$$

* momentum coefficient

$$
\beta=0.9,
$$

* previous velocity

$$
v=0.5,
$$

* current gradient

$$
\nabla J=2.
$$

The new velocity becomes

$$
\begin{aligned}
v
&=
0.9(0.5)
-
0.1(2)\\
&=
0.45-0.20\\
&=
0.25.
\end{aligned}
$$

If the current parameter value is

$$
w=5,
$$

then

$$
w_{\text{new}}
=
5+0.25
=
5.25.
$$

Unlike vanilla Gradient Descent,

the update reflects both

the current gradient

and

the accumulated history.

---

# <font color='orange'> 10. Advantages of Momentum </font>

Momentum offers several benefits.

* Faster convergence.
* Reduced oscillations.
* Improved stability.
* Better navigation through shallow valleys.
* Simple extension of Gradient Descent.

These advantages explain why Momentum remains widely used in modern deep learning.

---

# <font color='orange'> 11. Limitations </font>

Momentum also has limitations.

* Requires choosing the momentum coefficient.
* Can overshoot the optimum if the learning rate is too large.
* Still uses a single learning rate for all parameters.
* Does not adapt updates to individual parameters.

These limitations motivate adaptive optimization methods such as AdaGrad, RMSProp, and Adam.

---

# <font color='orange'> 12. Practical Implementation in TensorFlow </font>

Momentum is easily enabled in TensorFlow.

```python
import tensorflow as tf

optimizer = tf.keras.optimizers.SGD(

    learning_rate=0.01,

    momentum=0.9

)
```

Changing from ordinary SGD to Momentum requires only the additional `momentum` argument.

---

# <font color='orange'> 13. Common Misconceptions </font>

### Momentum Changes the Learning Rate

False.

The learning rate remains

\(\eta\).

Momentum introduces

an additional velocity term,

not a new learning rate.

---

### Larger Momentum Is Always Better

False.

Extremely large values of

\(\beta\)

may cause the optimizer to overshoot the minimum or converge more slowly.

Typical values between

0.9 and 0.99

work well in many applications.

---

### Momentum Eliminates the Need for Other Optimizers

False.

Momentum significantly improves Gradient Descent but does not address parameter-specific learning rates, which motivates later optimizers such as RMSProp and Adam.

---

# <font color='purple'> 14. Conceptual Summary </font>

| Concept | Description |
| :--- | :--- |
| Momentum | Extension of Gradient Descent using previous updates |
| Velocity | Running average of previous gradients |
| Momentum Coefficient (\(\beta\)) | Controls how much previous velocity is retained |
| Main Benefit | Faster convergence with reduced oscillations |
| Typical Value of \(\beta\) | 0.9 or 0.99 |
| Limitation | Uses a single learning rate for all parameters |
| Main Goal | Accelerate optimization while smoothing parameter updates |

> **Key Insight:** Momentum enhances Gradient Descent by introducing a velocity term that accumulates information from previous parameter updates. Rather than responding only to the current gradient, the optimizer builds momentum along consistently beneficial directions while damping oscillations in steep regions of the loss landscape. This leads to faster and more stable convergence, making Momentum one of the foundational improvements in modern optimization algorithms and an important precursor to methods such as Nesterov Momentum and Adam.

# <font color='blue'> Nesterov Accelerated Gradient (NAG) </font>

Momentum optimization improves Gradient Descent by accumulating information from previous parameter updates, enabling faster convergence and reducing oscillations in narrow valleys. However, classical Momentum computes the gradient at the **current parameter values**, even though the momentum term will subsequently move the parameters to a different location. As a result, the optimizer may occasionally overshoot the optimum or make less informed updates.

**Nesterov Accelerated Gradient (NAG)** addresses this limitation by computing the gradient at an estimated future position rather than at the current location. By "looking ahead" before evaluating the gradient, NAG anticipates where the optimizer is heading and adjusts its trajectory accordingly. This often leads to faster convergence and improved stability compared with classical Momentum.

This chapter introduces the motivation behind Nesterov Momentum, develops its mathematical formulation, compares it with classical Momentum, and discusses its practical implementation in modern deep learning frameworks.

---

# <font color='orange'> 1. The Limitation of Classical Momentum </font>

Recall the Momentum update equations

$$
\boxed{
\mathbf{v}
=
\beta\mathbf{v}
-
\eta\nabla J(\mathbf{W})
}
$$

$$
\boxed{
\mathbf{W}
\leftarrow
\mathbf{W}
+
\mathbf{v}
}
$$

Notice that

the gradient

is computed

at the **current**

parameter values.

The optimizer therefore decides where to move

before considering

where momentum will actually carry it.

---

# <font color='orange'> 2. An Everyday Analogy </font>

Imagine driving a car.

Classical Momentum is like

looking only at

your current position

before steering.

Nesterov Momentum is like

looking a short distance

ahead on the road

before deciding how to steer.

Looking ahead usually produces

smoother

and

more accurate

navigation.

---

# <font color='orange'> 3. The Central Idea </font>

Instead of computing

the gradient at

$$
\mathbf{W},
$$

NAG first estimates

the future position,

$$
\boxed{
\mathbf{W}_{\text{look-ahead}}
=
\mathbf{W}
+
\beta\mathbf{v}.
}
$$

The gradient is then evaluated

at this estimated location.

Conceptually,

```
Current Position

↓

Estimated Future Position

↓

Compute Gradient

↓

Update Velocity

↓

Update Parameters
```

This simple modification improves the quality of each update.

---

# <font color='orange'> 4. Mathematical Formulation </font>

The look-ahead position is

$$
\boxed{
\tilde{\mathbf{W}}
=
\mathbf{W}
+
\beta\mathbf{v}.
}
$$

The velocity update becomes

$$
\boxed{
\mathbf{v}
=
\beta\mathbf{v}
-
\eta
\nabla J(\tilde{\mathbf{W}})
}
$$

Finally,

the parameters are updated

using

$$
\boxed{
\mathbf{W}
\leftarrow
\mathbf{W}
+
\mathbf{v}.
}
$$

The only difference from classical Momentum is

where

the gradient is evaluated.

---

# <font color='orange'> 5. Why Looking Ahead Helps </font>

Suppose the optimizer is moving rapidly toward a minimum.

Classical Momentum computes

the gradient

before

the momentum step.

NAG computes

the gradient

after estimating

where the momentum step will lead.

Consequently,

the optimizer receives earlier warning

when approaching steep regions,

allowing it to reduce overshooting.

---

# <font color='orange'> 6. Geometric Interpretation </font>

Conceptually,

Classical Momentum behaves as

```
Current Point

↓

Gradient

↓

Move
```

Nesterov Momentum behaves as

```
Current Point

↓

Predict Next Position

↓

Gradient There

↓

Move
```

Because the gradient is evaluated closer to the future trajectory,

the resulting update is often more accurate.

---

# <font color='orange'> 7. Comparison with Classical Momentum </font>

| Feature | Momentum | NAG |
| :--- | :---: | :---: |
| Uses Previous Velocity | ✓ | ✓ |
| Looks Ahead | ✗ | ✓ |
| Computes Gradient at Current Position | ✓ | ✗ |
| Often Converges Faster | Moderate | Better |
| Reduces Overshooting | Moderate | Better |

NAG retains all the advantages of Momentum while providing a more informed estimate of the descent direction.

---

# <font color='orange'> 8. Advantages </font>

Nesterov Momentum offers several practical benefits.

* Faster convergence.
* Better anticipation of the loss landscape.
* Reduced overshooting.
* Improved optimization stability.
* Simple modification of classical Momentum.

For many optimization problems,

NAG performs slightly better than standard Momentum.

---

# <font color='orange'> 9. Limitations </font>

Despite its improvements,

NAG still has several limitations.

* Requires selecting the learning rate.
* Requires selecting the momentum coefficient.
* Uses a single learning rate for all parameters.
* Does not automatically adapt to differences between parameters.

These limitations motivate adaptive optimization algorithms such as AdaGrad and RMSProp.

---

# <font color='orange'> 10. Practical Implementation in TensorFlow </font>

TensorFlow enables Nesterov Momentum through the `nesterov` argument.

```python
import tensorflow as tf

optimizer = tf.keras.optimizers.SGD(

    learning_rate=0.01,

    momentum=0.9,

    nesterov=True

)
```

Only one additional argument is required beyond standard Momentum.

---

# <font color='orange'> 11. When Should NAG Be Used? </font>

NAG is often preferred over classical Momentum when

* faster convergence is desired,
* optimization landscapes contain narrow valleys,
* Momentum alone exhibits noticeable overshooting.

Although Adam has become the default optimizer for many deep learning applications,

NAG remains widely used,

particularly in some computer vision and large-scale optimization tasks.

---

# <font color='orange'> 12. Common Misconceptions </font>

### NAG Is a Completely Different Optimizer

False.

NAG is an extension of classical Momentum.

The only conceptual difference is

where the gradient is evaluated.

---

### NAG Eliminates Overshooting Completely

False.

Although NAG generally reduces overshooting,

it cannot eliminate it entirely,

particularly when the learning rate is too large.

---

### NAG Automatically Chooses the Learning Rate

False.

The learning rate remains a user-defined hyperparameter.

NAG improves the update direction,

not the learning rate itself.

---

# <font color='purple'> 13. Conceptual Summary </font>

| Concept | Description |
| :--- | :--- |
| Nesterov Accelerated Gradient | Improvement of Momentum using a look-ahead gradient |
| Look-Ahead Position | Estimated future parameter values before computing the gradient |
| Velocity | Running average of previous updates |
| Main Innovation | Computes the gradient at the predicted future position |
| Main Advantage | Faster convergence with reduced overshooting |
| Main Limitation | Still uses a single global learning rate |
| Historical Importance | Bridge between Momentum methods and adaptive optimization algorithms |

> **Key Insight:** Nesterov Accelerated Gradient refines classical Momentum by evaluating the gradient at an estimated future position rather than at the current parameter values. This "look-ahead" strategy allows the optimizer to anticipate changes in the loss landscape, resulting in faster convergence and improved stability. Although NAG retains the same momentum-based philosophy as classical Momentum, its more informed gradient estimates often produce smoother optimization trajectories and better overall performance.

# <font color='blue'> AdaGrad (Adaptive Gradient Algorithm) </font>

Gradient Descent and its momentum-based variants use a single global learning rate for all model parameters. While this approach is simple and effective for many problems, it assumes that every parameter should be updated by approximately the same amount. In practice, however, different parameters often exhibit very different gradient magnitudes and learning dynamics.

The **Adaptive Gradient Algorithm (AdaGrad)** addresses this limitation by assigning an independent learning rate to every parameter. Parameters that receive large gradients have their effective learning rates reduced over time, whereas parameters that receive small or infrequent gradients retain relatively larger learning rates. This adaptive behaviour makes AdaGrad particularly effective for problems involving sparse data and high-dimensional feature spaces.

This chapter introduces the motivation behind adaptive learning rates, derives the AdaGrad update equations, explains its advantages and limitations, and discusses its historical significance in the evolution of modern optimization algorithms.

---

# <font color='orange'> 1. Why One Learning Rate Can Be Limiting </font>

Recall the Gradient Descent update rule

$$
\boxed{
W
\leftarrow
W
-
\eta
\nabla J
}
$$

The learning rate

\(\eta\)

is identical

for every parameter.

Conceptually,

```
Weight 1

↓

η = 0.01
```

```
Weight 2

↓

η = 0.01
```

```
Weight 3

↓

η = 0.01
```

However,

different parameters often require different step sizes during optimization.

---

# <font color='orange'> 2. Motivation for Adaptive Learning Rates </font>

Suppose one parameter consistently receives

very large gradients,

while another receives

very small gradients.

Ideally,

we would like

```
Large Gradient

↓

Smaller Learning Rate
```

and

```
Small Gradient

↓

Larger Learning Rate
```

rather than treating every parameter identically.

AdaGrad achieves precisely this objective.

---

# <font color='orange'> 3. The Central Idea </font>

AdaGrad keeps track of

how much each parameter has changed in the past.

Parameters with

large accumulated gradients

receive

smaller future updates.

Parameters with

small accumulated gradients

continue learning relatively quickly.

Thus,

every parameter develops

its own effective learning rate.

---

# <font color='orange'> 4. Accumulating Squared Gradients </font>

For each parameter,

AdaGrad maintains

an accumulated sum of squared gradients.

$$
\boxed{
G_t
=
G_{t-1}
+
g_t^2
}
$$

where

* \(g_t\) is the current gradient,
* \(G_t\) stores the cumulative squared gradients.

Because gradients are squared,

both positive and negative values contribute positively.

---

# <font color='orange'> 5. The AdaGrad Update Rule </font>

The parameter update becomes

$$
\boxed{
W
\leftarrow
W
-
\frac{\eta}
{\sqrt{G_t+\varepsilon}}
g_t
}
$$

where

* \(\eta\) is the initial learning rate,
* \(G_t\) is the accumulated squared gradient,
* \(\varepsilon\) is a small constant preventing division by zero.

Notice that

the denominator increases over time,

causing the effective learning rate to decrease automatically.

---

# <font color='orange'> 6. Interpreting the Update </font>

Suppose

a parameter has experienced

large gradients

for many iterations.

Then

$$
G_t
$$

becomes large,

making

$$
\frac{\eta}
{\sqrt{G_t}}
$$

small.

Consequently,

future updates become smaller.

Conversely,

parameters with infrequent or small gradients retain relatively larger update steps.

---

# <font color='orange'> 7. Why AdaGrad Works Well for Sparse Data </font>

Many machine learning problems contain

**sparse features**,

where only a small subset of features is active for each training example.

Examples include

* natural language processing,
* recommendation systems,
* text classification.

Parameters corresponding to rarely occurring features accumulate gradients slowly,

allowing them to continue receiving relatively large updates.

This makes AdaGrad particularly effective for sparse learning problems.

---

# <font color='orange'> 8. The Diminishing Learning Rate Problem </font>

AdaGrad's greatest strength eventually becomes its greatest weakness.

Since

$$
G_t
$$

only increases,

the effective learning rate

continually decreases.

Conceptually,

```
Large Learning Rate

↓

Medium Learning Rate

↓

Small Learning Rate

↓

Tiny Learning Rate
```

Eventually,

parameter updates may become so small

that learning effectively stops,

even though the model has not yet reached an optimal solution.

This limitation motivated the development of RMSProp.

---

# <font color='orange'> 9. Advantages of AdaGrad </font>

AdaGrad offers several important benefits.

* Automatically adapts learning rates.
* Requires little manual tuning.
* Excellent performance on sparse datasets.
* Simple mathematical formulation.
* Introduced adaptive optimization to deep learning.

These ideas influenced nearly all subsequent adaptive optimizers.

---

# <font color='orange'> 10. Limitations of AdaGrad </font>

AdaGrad also has significant limitations.

* Learning rates decrease monotonically.
* Training may stop prematurely.
* Less effective for long training runs.
* Often outperformed by newer adaptive optimizers.

These limitations explain why AdaGrad is now used less frequently than RMSProp or Adam.

---

# <font color='orange'> 11. Practical Implementation in TensorFlow </font>

TensorFlow provides a built-in implementation.

```python
import tensorflow as tf

optimizer = tf.keras.optimizers.Adagrad(

    learning_rate=0.01

)
```

Only the optimizer needs to be changed;

the remainder of the training code remains identical.

---

# <font color='orange'> 12. Historical Importance </font>

AdaGrad introduced one of the most influential ideas in optimization:

> **Different parameters should learn at different rates.**

This concept directly inspired

* RMSProp,
* AdaDelta,
* Adam,
* AdamW,

which remain among the most widely used optimizers today.

---

# <font color='orange'> 13. Common Misconceptions </font>

### AdaGrad Chooses Completely Independent Learning Rates

Not exactly.

All parameters begin with the same initial learning rate,

but their **effective** learning rates evolve differently because each parameter accumulates its own squared gradients.

---

### AdaGrad Always Outperforms SGD

False.

AdaGrad performs exceptionally well for some sparse problems,

but its continually shrinking learning rates can make it inferior to RMSProp or Adam for many deep learning tasks.

---

### The Learning Rate Never Changes

False.

The user specifies only the initial learning rate.

AdaGrad automatically adjusts the effective learning rate for every parameter throughout training.

---

# <font color='purple'> 14. Conceptual Summary </font>

| Concept | Description |
| :--- | :--- |
| AdaGrad | Adaptive optimization algorithm with parameter-specific learning rates |
| Accumulated Gradient | Running sum of squared gradients |
| Effective Learning Rate | Automatically decreases as gradients accumulate |
| Main Advantage | Excellent performance for sparse features |
| Main Limitation | Learning rates decrease continually and may become too small |
| Historical Importance | First widely adopted adaptive optimizer |
| Main Goal | Adapt the learning rate individually for each parameter |

> **Key Insight:** AdaGrad introduced the powerful idea that different model parameters should not necessarily share the same learning rate. By accumulating squared gradients for each parameter individually, AdaGrad automatically reduces the effective learning rate for frequently updated parameters while preserving larger updates for infrequently updated ones. Although its continually decreasing learning rates eventually limit its effectiveness, AdaGrad established the adaptive optimization principles that underpin modern optimizers such as RMSProp and Adam.

# <font color='blue'> RMSProp (Root Mean Square Propagation) </font>

AdaGrad introduced the important idea of adaptive learning rates by assigning a separate effective learning rate to each model parameter. Although highly successful for sparse learning problems, AdaGrad continually accumulates squared gradients throughout training. As a consequence, the effective learning rates decrease monotonically and may eventually become so small that parameter updates nearly cease.

**Root Mean Square Propagation (RMSProp)** addresses this limitation by replacing AdaGrad's cumulative sum of squared gradients with an **exponentially weighted moving average** of recent squared gradients. Instead of remembering the entire optimization history, RMSProp emphasizes recent gradient information while gradually forgetting older updates. This allows the optimizer to retain adaptive learning rates without allowing them to shrink indefinitely.

This chapter introduces the motivation behind RMSProp, derives its update equations, explains exponentially weighted moving averages, compares RMSProp with AdaGrad, and discusses its practical implementation in modern deep learning frameworks.

---

# <font color='orange'> 1. Why AdaGrad Eventually Stops Learning </font>

Recall AdaGrad's accumulated gradient,

$$
\boxed{
G_t
=
G_{t-1}
+
g_t^2.
}
$$

Since

$$
G_t
$$

always increases,

the effective learning rate

$$
\frac{\eta}
{\sqrt{G_t}}
$$

continuously decreases.

Eventually,

parameter updates become

extremely small.

Learning therefore slows dramatically,

even when the network has not yet reached an optimal solution.

---

# <font color='orange'> 2. The Central Idea Behind RMSProp </font>

Instead of remembering

**all**

previous gradients,

RMSProp remembers only

the **recent** ones.

Older gradients are gradually forgotten.

Conceptually,

```
Recent Gradients

↓↓↓

Important

↓

Old Gradients

↓

Gradually Forgotten
```

This prevents the learning rate from shrinking indefinitely.

---

# <font color='orange'> 3. Exponentially Weighted Moving Average (EWMA) </font>

The key mathematical idea is the

**Exponentially Weighted Moving Average (EWMA).**

Instead of computing

a cumulative sum,

RMSProp computes

$$
\boxed{
S_t
=
\rho S_{t-1}
+
(1-\rho)g_t^2
}
$$

where

* \(g_t\) is the current gradient,
* \(S_t\) is the moving average of squared gradients,
* \(\rho\) is the decay rate.

Unlike AdaGrad,

older gradients receive progressively smaller weights.

---

# <font color='orange'> 4. Why Exponential Averaging Works </font>

Suppose

```
Gradient History

↓

Recent

↓

Old

↓

Very Old
```

EWMA assigns

```
Recent Gradient

↓

Large Weight
```

```
Old Gradient

↓

Small Weight
```

```
Very Old Gradient

↓

Tiny Weight
```

Consequently,

the optimizer adapts to the current optimization landscape

rather than the entire training history.

---

# <font color='orange'> 5. RMSProp Update Rule </font>

First,

update the moving average,

$$
\boxed{
S_t
=
\rho S_{t-1}
+
(1-\rho)g_t^2.
}
$$

Then,

update the parameter,

$$
\boxed{
W
\leftarrow
W
-
\frac{\eta}
{\sqrt{S_t+\varepsilon}}
g_t.
}
$$

where

* \(\eta\) is the learning rate,
* \(\varepsilon\) is a small constant preventing division by zero.

The update resembles AdaGrad,

but

\(S_t\)

remains bounded because older gradients gradually disappear.

---

# <font color='orange'> 6. The Decay Rate </font>

The parameter

\(\rho\)

controls

how quickly older gradients are forgotten.

Typical values are

```
ρ = 0.9

or

ρ = 0.99.
```

---

### Small ρ

```
Short Memory

↓

Responds Quickly
```

---

### Large ρ

```
Long Memory

↓

Smoother Updates
```

The choice of

\(\rho\)

balances stability against responsiveness.

---

# <font color='orange'> 7. AdaGrad vs RMSProp </font>

| Feature | AdaGrad | RMSProp |
| :--- | :---: | :---: |
| Adaptive Learning Rates | ✓ | ✓ |
| Stores Entire Gradient History | ✓ | ✗ |
| Uses Exponential Moving Average | ✗ | ✓ |
| Learning Rate Eventually Vanishes | ✓ | ✗ |
| Suitable for Long Training | Limited | Excellent |

RMSProp preserves AdaGrad's strengths while avoiding its principal weakness.

---

# <font color='orange'> 8. Why RMSProp Converges Faster </font>

Because

older gradients

gradually disappear,

the effective learning rate

remains approximately stable.

Conceptually,

```
AdaGrad

↓

Smaller

↓

Smaller

↓

Tiny
```

```
RMSProp

↓

Adaptive

↓

Adaptive

↓

Adaptive
```

The optimizer therefore continues making meaningful updates throughout training.

---

# <font color='orange'> 9. Advantages </font>

RMSProp provides several practical benefits.

* Adaptive learning rates.
* Suitable for long training runs.
* Faster convergence than AdaGrad.
* Handles noisy mini-batch gradients well.
* Widely used in deep learning.

These properties explain its continued popularity.

---

# <font color='orange'> 10. Limitations </font>

Despite its strengths,

RMSProp still has limitations.

* Requires selecting the learning rate.
* Requires selecting the decay rate.
* Does not explicitly use momentum.
* May be outperformed by Adam in many applications.

These limitations motivated the development of Adam.

---

# <font color='orange'> 11. Practical Implementation in TensorFlow </font>

TensorFlow provides a built-in implementation.

```python
import tensorflow as tf

optimizer = tf.keras.optimizers.RMSprop(

    learning_rate=0.001,

    rho=0.9

)
```

The optimizer automatically maintains the exponentially weighted moving averages during training.

---

# <font color='orange'> 12. Historical Importance </font>

RMSProp introduced one of the most influential ideas in modern optimization:

> **Forget old gradients and focus on recent optimization behaviour.**

This principle,

combined with Momentum,

led directly to

the Adam optimizer,

which remains one of the most widely used optimizers in deep learning.

---

# <font color='orange'> 13. Common Misconceptions </font>

### RMSProp Eliminates the Learning Rate

False.

RMSProp still requires an initial learning rate.

It adjusts the **effective** learning rate for each parameter during training.

---

### RMSProp Is Just AdaGrad with a Different Formula

False.

The crucial difference is the use of an exponentially weighted moving average rather than an ever-growing cumulative sum of squared gradients.

This fundamentally changes the long-term behaviour of the optimizer.

---

### RMSProp Includes Momentum

Not in its standard form.

Although TensorFlow allows an optional momentum parameter,

the classical RMSProp algorithm adapts learning rates but does not inherently include Momentum.

Adam combines both ideas explicitly.

---

# <font color='purple'> 14. Conceptual Summary </font>

| Concept | Description |
| :--- | :--- |
| RMSProp | Adaptive optimizer using recent squared gradients |
| Exponentially Weighted Moving Average | Running average that emphasizes recent gradients |
| Decay Rate (\(\rho\)) | Controls how quickly old gradients are forgotten |
| Main Advantage | Prevents learning rates from shrinking indefinitely |
| Main Limitation | Does not explicitly incorporate momentum |
| Historical Importance | Forms one half of the Adam optimizer |
| Main Goal | Maintain adaptive learning rates throughout training |

> **Key Insight:** RMSProp improves upon AdaGrad by replacing the cumulative sum of squared gradients with an exponentially weighted moving average. By emphasizing recent gradients while gradually forgetting older ones, RMSProp maintains adaptive learning rates without allowing them to decay toward zero. This enables efficient optimization over long training runs and provides one of the two fundamental ideas—along with Momentum—that are combined in the Adam optimizer.

# <font color='blue'> Adam (Adaptive Moment Estimation) </font>

The optimization algorithms studied thus far have each addressed different shortcomings of standard Gradient Descent. Momentum accelerates convergence by accumulating information from previous gradients, while RMSProp improves optimization by adapting the learning rate for each parameter according to recent gradient magnitudes. Both methods significantly improve training performance, yet each addresses only one aspect of the optimization problem.

**Adam (Adaptive Moment Estimation)** combines these two complementary ideas into a single optimization algorithm. It simultaneously maintains an exponentially weighted moving average of the gradients (first moment) and an exponentially weighted moving average of the squared gradients (second moment). By combining momentum with adaptive learning rates, Adam achieves rapid convergence, stable optimization, and excellent performance across a wide range of deep learning problems.

Since its introduction by Diederik Kingma and Jimmy Ba in 2014, Adam has become one of the most widely used optimization algorithms in machine learning and deep learning.

---

# <font color='orange'> 1. Why Do We Need Adam? </font>

Momentum solves

```
Slow Convergence
```

by remembering previous gradients.

RMSProp solves

```
Shrinking Learning Rates
```

by adapting learning rates for each parameter.

A natural question arises:

> **Why not combine both ideas?**

Adam is precisely this combination.

---

# <font color='orange'> 2. The Two Key Ideas </font>

Adam maintains two separate moving averages.

### First Moment

Tracks

the average gradient.

Conceptually,

```
Current Gradient

↓

Running Average

↓

Momentum
```

---

### Second Moment

Tracks

the average squared gradient.

Conceptually,

```
Squared Gradient

↓

Running Average

↓

Adaptive Learning Rate
```

Together,

these two quantities determine each parameter update.

---

# <font color='orange'> 3. First Moment Estimate </font>

Adam computes an exponentially weighted moving average of the gradients.

$$
\boxed{
m_t
=
\beta_1m_{t-1}
+
(1-\beta_1)g_t
}
$$

where

* \(g_t\) is the current gradient,
* \(m_t\) is the first moment estimate,
* \(\beta_1\) controls how quickly previous gradients are forgotten.

This is mathematically equivalent to the Momentum algorithm.

---

# <font color='orange'> 4. Second Moment Estimate </font>

Adam also computes an exponentially weighted moving average of the squared gradients.

$$
\boxed{
v_t
=
\beta_2v_{t-1}
+
(1-\beta_2)g_t^2
}
$$

where

* \(v_t\) is the second moment estimate,
* \(\beta_2\) controls the averaging of squared gradients.

This is closely related to RMSProp.

---

# <font color='orange'> 5. Why Bias Correction is Needed </font>

At the beginning of training,

both

\(m_t\)

and

\(v_t\)

are initialized to zero.

Consequently,

their early values are biased toward zero.

Adam corrects this initialization bias using

$$
\boxed{
\hat m_t
=
\frac{m_t}
{1-\beta_1^t}
}
$$

and

$$
\boxed{
\hat v_t
=
\frac{v_t}
{1-\beta_2^t}
}
$$

These **bias-corrected estimates** provide more accurate gradient statistics, especially during the first few optimization steps.

---

# <font color='orange'> 6. Adam Parameter Update </font>

The final parameter update is

$$
\boxed{
W
\leftarrow
W
-
\eta
\frac{\hat m_t}
{\sqrt{\hat v_t}+\varepsilon}
}
$$

where

* \(\eta\) is the learning rate,
* \(\hat m_t\) is the bias-corrected first moment,
* \(\hat v_t\) is the bias-corrected second moment,
* \(\varepsilon\) prevents division by zero.

This single equation combines the advantages of Momentum and RMSProp.

---

# <font color='orange'> 7. Typical Hyperparameters </font>

The default values proposed by Kingma and Ba are

| Parameter | Typical Value | Purpose |
| :--- | :---: | :--- |
| Learning Rate (\(\eta\)) | 0.001 | Overall step size |
| \(\beta_1\) | 0.9 | First moment decay |
| \(\beta_2\) | 0.999 | Second moment decay |
| \(\varepsilon\) | \(10^{-8}\) | Numerical stability |

These defaults work well for many deep learning applications.

---

# <font color='orange'> 8. Why Adam Works Well </font>

Adam combines several desirable properties.

* Accelerates learning through momentum.
* Adapts learning rates individually for each parameter.
* Handles noisy mini-batch gradients effectively.
* Requires relatively little hyperparameter tuning.
* Performs well across many different architectures and datasets.

These features explain its widespread adoption.

---

# <font color='orange'> 9. Advantages </font>

Adam offers several important benefits.

* Fast convergence.
* Stable optimization.
* Adaptive learning rates.
* Robust performance on large datasets.
* Effective for sparse gradients.
* Suitable for deep neural networks.

For many practitioners,

Adam is the first optimizer to try.

---

# <font color='orange'> 10. Limitations </font>

Despite its popularity,

Adam is not universally optimal.

* May generalize less well than SGD with Momentum on some vision tasks.
* More computationally expensive than SGD.
* Can converge to sharp minima in certain settings.
* Weight decay must be handled carefully.

These observations motivated improved variants such as AdamW.

---

# <font color='orange'> 11. Adam vs Previous Optimizers </font>

| Optimizer | Momentum | Adaptive Learning Rate |
| :--- | :---: | :---: |
| Gradient Descent | ✗ | ✗ |
| Momentum | ✓ | ✗ |
| NAG | ✓ | ✗ |
| AdaGrad | ✗ | ✓ |
| RMSProp | ✗ | ✓ |
| **Adam** | ✓ | ✓ |

Adam combines the two major optimization ideas introduced earlier.

---

# <font color='orange'> 12. Practical Implementation in TensorFlow </font>

TensorFlow provides a built-in implementation.

```python
import tensorflow as tf

optimizer = tf.keras.optimizers.Adam(

    learning_rate=0.001,

    beta_1=0.9,

    beta_2=0.999,

    epsilon=1e-8

)
```

In many applications,

the default hyperparameters require little or no modification.

---

# <font color='orange'> 13. Common Misconceptions </font>

### Adam Always Produces the Best Model

False.

Although Adam often converges faster,

SGD with Momentum can sometimes achieve better final generalization, particularly in image classification.

---

### Adam Eliminates the Need to Tune the Learning Rate

False.

Adam is generally less sensitive to the learning rate than SGD, but selecting an appropriate learning rate can still significantly affect training performance.

---

### Adam Is Simply Momentum Plus RMSProp

Not exactly.

Adam combines the key ideas behind Momentum and RMSProp, **but it also introduces bias correction**, which is essential for obtaining reliable parameter updates during the early stages of training.

---

# <font color='orange'> 14. Historical Importance </font>

Adam has become one of the most influential optimization algorithms in deep learning.

It is widely used in

* computer vision,
* natural language processing,
* speech recognition,
* reinforcement learning,
* generative AI,
* scientific machine learning.

Its combination of speed, stability, and ease of use has made it the default optimizer in many deep learning frameworks.

---

# <font color='purple'> 15. Conceptual Summary </font>

| Concept | Description |
| :--- | :--- |
| Adam | Adaptive Moment Estimation optimizer |
| First Moment | Exponentially weighted moving average of gradients (Momentum) |
| Second Moment | Exponentially weighted moving average of squared gradients (RMSProp) |
| Bias Correction | Corrects initialization bias in the moving averages |
| Main Advantage | Combines momentum with adaptive learning rates |
| Typical Hyperparameters | \(\eta=0.001\), \(\beta_1=0.9\), \(\beta_2=0.999\) |
| Main Limitation | May not always provide the best generalization performance |
| Historical Importance | One of the most widely used optimizers in modern deep learning |

> **Key Insight:** Adam integrates the complementary strengths of Momentum and RMSProp into a single optimization algorithm. By maintaining exponentially weighted moving averages of both the gradients and their squared values, together with bias correction, Adam achieves fast convergence, adaptive parameter updates, and robust optimization across a wide range of deep learning tasks. Its efficiency, stability, and ease of use have made it one of the standard optimizers for training modern neural networks.

# <font color='blue'> AdamW: Adam with Decoupled Weight Decay </font>

Regularization plays a central role in deep learning by reducing overfitting and improving the generalization ability of neural networks. One of the most widely used regularization techniques is **L2 regularization**, which penalizes large parameter values by adding a quadratic penalty to the loss function. For standard Gradient Descent, L2 regularization and **weight decay** are mathematically equivalent and produce identical parameter updates.

For adaptive optimization algorithms such as Adam, however, this equivalence no longer holds. Because Adam rescales parameter updates using adaptive learning rates, incorporating L2 regularization directly into the gradient affects the optimization dynamics in unintended ways. **AdamW** resolves this issue by **decoupling weight decay from the gradient update**, allowing regularization and optimization to operate independently.

Introduced by Loshchilov and Hutter (2019), AdamW has become the preferred optimizer for many modern deep learning architectures, including transformers and large language models.

---

# <font color='orange'> 1. Review of L2 Regularization </font>

Recall that L2 regularization modifies the loss function by adding a penalty term,

$$
\boxed{
J_{\text{total}}
=
J
+
\frac{\lambda}{2}
\sum_i
W_i^2
}
$$

where

* \(J\) is the original loss,
* \(\lambda\) controls the strength of regularization.

This encourages smaller parameter values,

reducing overfitting.

---

# <font color='orange'> 2. What is Weight Decay? </font>

Weight decay shrinks the parameters directly during each optimization step.

Instead of only following the gradient,

the parameters are also multiplied by a factor slightly smaller than one.

Conceptually,

```
Large Weights

↓

Shrink Slightly

↓

Repeat
```

Over time,

this discourages unnecessarily large parameter values.

---

# <font color='orange'> 3. Why L2 and Weight Decay Are Equivalent for SGD </font>

For standard Gradient Descent,

the update rule with L2 regularization becomes

$$
\boxed{
W
\leftarrow
W
-
\eta
\left(
\nabla J
+
\lambda W
\right)
}
$$

Rearranging,

$$
\boxed{
W
=
(1-\eta\lambda)W
-
\eta\nabla J
}
$$

Notice that

the factor

$$
(1-\eta\lambda)
$$

simply shrinks the weights.

Thus,

for SGD,

L2 regularization

and

weight decay

are mathematically identical.

---

# <font color='orange'> 4. Why Adam is Different </font>

Adam updates parameters using

adaptive learning rates.

The gradient is divided by

$$
\sqrt{\hat v_t}
$$

before the parameter update.

If L2 regularization is added directly to the gradient,

its effect is also scaled by the adaptive learning rate.

Consequently,

different parameters experience different amounts of regularization.

This is **not** true weight decay.

---

# <font color='orange'> 5. The Central Idea of AdamW </font>

AdamW separates

optimization

from

regularization.

Conceptually,

```
Gradient

↓

Adam Update

↓

Weight Decay

↓

Final Parameters
```

Instead of including

\(\lambda W\)

inside the gradient,

AdamW applies weight decay

after

the adaptive update.

Hence the name

**decoupled weight decay**.

---

# <font color='orange'> 6. AdamW Update Rule </font>

Adam first computes the adaptive update,

$$
\boxed{
W
\leftarrow
W
-
\eta
\frac{\hat m_t}
{\sqrt{\hat v_t}+\varepsilon}
}
$$

AdamW then applies weight decay separately,

$$
\boxed{
W
\leftarrow
W
-
\eta\lambda W
}
$$

Combining the two gives

$$
\boxed{
W
\leftarrow
W
-
\eta
\frac{\hat m_t}
{\sqrt{\hat v_t}+\varepsilon}
-
\eta\lambda W.
}
$$

Notice that

the weight decay term

is **not divided**

by

$$
\sqrt{\hat v_t}.
$$

This preserves the intended regularization behaviour.

---

# <font color='orange'> 7. Adam vs AdamW </font>

| Feature | Adam | AdamW |
| :--- | :---: | :---: |
| Adaptive Learning Rates | ✓ | ✓ |
| Momentum | ✓ | ✓ |
| Bias Correction | ✓ | ✓ |
| Decoupled Weight Decay | ✗ | ✓ |
| Better Generalization | Sometimes | Often |

The primary distinction is how regularization is applied.

---

# <font color='orange'> 8. Why AdamW Generalizes Better </font>

Separating optimization

from

regularization

allows each component

to perform its intended role independently.

Benefits include

* more consistent regularization,
* improved generalization,
* more stable optimization,
* easier hyperparameter tuning.

These advantages become increasingly important for very large neural networks.

---

# <font color='orange'> 9. Typical Hyperparameters </font>

AdamW generally uses the same defaults as Adam.

| Parameter | Typical Value |
| :--- | :---: |
| Learning Rate | 0.001 |
| \(\beta_1\) | 0.9 |
| \(\beta_2\) | 0.999 |
| \(\varepsilon\) | \(10^{-8}\) |
| Weight Decay | \(10^{-2}\) to \(10^{-4}\) |

The optimal weight decay depends on the dataset and model architecture.

---

# <font color='orange'> 10. Practical Implementation in TensorFlow </font>

TensorFlow provides a built-in implementation.

```python
import tensorflow as tf

optimizer = tf.keras.optimizers.AdamW(

    learning_rate=0.001,

    weight_decay=1e-4

)
```

Only the optimizer changes;

the remainder of the training workflow remains unchanged.

---

# <font color='orange'> 11. Applications </font>

AdamW has become the optimizer of choice for many modern architectures.

Examples include

* Vision Transformers (ViTs),
* BERT,
* GPT-family language models,
* diffusion models,
* large-scale image classification,
* scientific deep learning.

Many contemporary deep learning libraries now recommend AdamW over standard Adam.

---

# <font color='orange'> 12. Common Misconceptions </font>

### AdamW is Simply Adam with L2 Regularization

False.

AdamW applies **decoupled weight decay**, whereas standard Adam incorporates L2 regularization into the gradient.

These approaches are mathematically equivalent for SGD but not for adaptive optimizers such as Adam.

---

### AdamW Always Outperforms Adam

Not necessarily.

For some problems,

the difference is small.

However,

AdamW often provides better generalization,

particularly for transformer-based models and large neural networks.

---

### Weight Decay Replaces Other Regularization Techniques

False.

Weight decay complements techniques such as

* dropout,
* early stopping,
* data augmentation,

rather than replacing them.

---

# <font color='orange'> 13. Historical Importance </font>

AdamW represents an important refinement of adaptive optimization.

Its introduction clarified a long-standing misconception regarding L2 regularization and weight decay in adaptive optimizers.

Today,

AdamW has become one of the standard optimizers for training state-of-the-art deep learning models.

---

# <font color='purple'> 14. Conceptual Summary </font>

| Concept | Description |
| :--- | :--- |
| AdamW | Adam optimizer with decoupled weight decay |
| Weight Decay | Direct shrinking of parameters during optimization |
| Decoupling | Separates regularization from gradient computation |
| Main Advantage | More effective regularization for adaptive optimizers |
| Typical Applications | Transformers, large language models, vision models |
| Main Goal | Improve generalization while retaining Adam's optimization strengths |

> **Key Insight:** AdamW extends Adam by separating weight decay from the adaptive gradient update. While L2 regularization and weight decay are equivalent for standard Gradient Descent, this equivalence breaks down for adaptive optimizers such as Adam. By decoupling these two processes, AdamW preserves the intended regularization effect, leading to improved generalization and making it one of the preferred optimizers for modern deep learning architectures.

# <font color='blue'> Learning Rate Scheduling </font>

The learning rate is one of the most influential hyperparameters in deep learning. It determines the size of each parameter update during optimization and therefore has a profound impact on training speed, numerical stability, and the final performance of the model. Although modern optimization algorithms such as Adam and AdamW automatically adapt parameter updates, they still require an initial learning rate whose value strongly influences the optimization process.

A fixed learning rate is rarely optimal throughout training. Large learning rates promote rapid progress during the early stages of optimization but may prevent convergence near the optimum. Conversely, small learning rates improve fine-tuning but may lead to excessively slow training. **Learning rate scheduling** addresses this problem by systematically modifying the learning rate as training progresses.

This chapter introduces the motivation for learning rate schedules, presents the most widely used scheduling strategies, discusses their practical implementation, and explains how learning rate scheduling improves optimization efficiency.

---

# <font color='orange'> 1. Why Does the Learning Rate Matter? </font>

Recall the parameter update equation

$$
\boxed{
W
\leftarrow
W
-
\eta
\nabla J
}
$$

where

\(\eta\)

is the learning rate.

The learning rate controls

how far

the optimizer moves

during each update.

---

# <font color='orange'> 2. Problems with a Fixed Learning Rate </font>

Suppose

```
η = 0.01
```

throughout training.

Early in optimization,

this value may be

too small,

leading to unnecessarily slow progress.

Later,

the same value may be

too large,

causing oscillations around the optimum.

A single fixed learning rate therefore represents a compromise rather than an ideal choice.

---

# <font color='orange'> 3. The Central Idea </font>

Instead of keeping

\(\eta\)

constant,

we allow it to change during training.

Conceptually,

```
Beginning

↓

Large Learning Rate

↓

Fast Progress

↓

Smaller Learning Rate

↓

Fine-Tuning
```

This generally improves both convergence speed and final model performance.

---

# <font color='orange'> 4. Step Decay </font>

One of the simplest schedules is

**Step Decay**.

The learning rate is reduced

after a fixed number of epochs.

Example

| Epoch | Learning Rate |
| :---: | :---: |
| 1–10 | 0.01 |
| 11–20 | 0.001 |
| 21–30 | 0.0001 |

Advantages

* Simple.
* Easy to implement.

Limitations

* Abrupt changes may temporarily disrupt optimization.

---

# <font color='orange'> 5. Exponential Decay </font>

Instead of changing the learning rate abruptly,

it may be reduced smoothly.

$$
\boxed{
\eta_t
=
\eta_0
e^{-kt}
}
$$

where

* \(\eta_0\) is the initial learning rate,
* \(k\) controls the decay rate,
* \(t\) denotes training progress.

This produces a gradual reduction in learning rate.

---

# <font color='orange'> 6. Polynomial Decay </font>

Polynomial decay reduces the learning rate according to

$$
\boxed{
\eta_t
=
\eta_0
\left(
1-
\frac{t}{T}
\right)^p
}
$$

where

* \(T\) is the total number of training steps,
* \(p\) determines the shape of the decay.

Polynomial schedules are commonly used in large-scale deep learning applications.

---

# <font color='orange'> 7. Cosine Annealing </font>

Cosine annealing gradually decreases the learning rate following a cosine curve.

Conceptually,

```
High

↓

Smooth Curve

↓

Very Small
```

Unlike exponential decay,

the reduction is slow at the beginning,

faster in the middle,

and gentle again near the end.

Cosine schedules often produce excellent empirical performance.

---

# <font color='orange'> 8. Warm Restarts </font>

Instead of continually decreasing the learning rate,

the optimizer can periodically reset it to a larger value.

Conceptually,

```
Large

↓

Small

↓

Restart

↓

Large

↓

Small
```

This strategy,

known as

**Stochastic Gradient Descent with Warm Restarts (SGDR)**,

may help the optimizer escape undesirable regions of the loss landscape.

---

# <font color='orange'> 9. Learning Rate Warm-Up </font>

Large learning rates at the very beginning of training can sometimes destabilize optimization,

particularly for deep networks and transformer models.

Warm-up addresses this by

starting with

a very small learning rate

and gradually increasing it.

Conceptually,

```
Very Small

↓

Small

↓

Normal

↓

Decay
```

Warm-up has become standard practice for training many transformer-based architectures.

---

# <font color='orange'> 10. Reduce-on-Plateau </font>

Instead of using a predetermined schedule,

the learning rate can be adjusted automatically.

The optimizer monitors

the validation loss.

If performance stops improving,

the learning rate is reduced.

Conceptually,

```
Validation Loss Stops Improving

↓

Reduce Learning Rate

↓

Continue Training
```

This adaptive strategy is widely used in practical deep learning workflows.

---

# <font color='orange'> 11. Comparison of Learning Rate Schedules </font>

| Schedule | Main Idea | Typical Use |
| :--- | :--- | :--- |
| Fixed | Constant learning rate | Simple experiments |
| Step Decay | Reduce after fixed intervals | Classical CNN training |
| Exponential Decay | Continuous reduction | General optimization |
| Polynomial Decay | Controlled nonlinear reduction | Large-scale training |
| Cosine Annealing | Cosine-shaped decay | Modern deep learning |
| Warm Restarts | Periodically increase learning rate | Escape poor local regions |
| Warm-Up | Gradually increase learning rate initially | Transformers |
| Reduce-on-Plateau | Reduce when validation performance stagnates | General-purpose training |

---

# <font color='orange'> 12. Practical Implementation in TensorFlow </font>

### Exponential Decay

```python
import tensorflow as tf

lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(

    initial_learning_rate=0.001,

    decay_steps=10000,

    decay_rate=0.96

)

optimizer = tf.keras.optimizers.Adam(

    learning_rate=lr_schedule

)
```

---

### Reduce-on-Plateau

```python
callback = tf.keras.callbacks.ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=5

)
```

The callback automatically reduces the learning rate when the validation loss ceases to improve.

---

# <font color='orange'> 13. Choosing a Learning Rate Schedule </font>

General recommendations include

| Situation | Recommended Schedule |
| :--- | :--- |
| Small neural networks | Fixed or Step Decay |
| CNNs | Step Decay or Cosine Annealing |
| Transformers | Warm-Up followed by Cosine Decay |
| Unknown optimization behaviour | Reduce-on-Plateau |

The optimal schedule depends on the model architecture, dataset, and computational budget.

---

# <font color='orange'> 14. Common Misconceptions </font>

### Adam Eliminates the Need for Learning Rate Scheduling

False.

Although Adam adapts parameter updates, the global learning rate remains an important hyperparameter, and scheduling it often improves performance.

---

### The Learning Rate Should Always Decrease

Not necessarily.

Warm-up deliberately increases the learning rate during the initial stages of training, and warm restart methods periodically increase it throughout training.

---

### More Complex Schedules Always Perform Better

False.

For many applications, a simple Step Decay or Reduce-on-Plateau schedule performs as well as more sophisticated alternatives.

---

# <font color='purple'> 15. Conceptual Summary </font>

| Concept | Description |
| :--- | :--- |
| Learning Rate Schedule | Strategy for modifying the learning rate during training |
| Step Decay | Reduces the learning rate at fixed intervals |
| Exponential Decay | Smooth exponential reduction |
| Polynomial Decay | Nonlinear reduction over training |
| Cosine Annealing | Cosine-shaped learning rate schedule |
| Warm Restarts | Periodic increases in the learning rate |
| Warm-Up | Gradual increase in the learning rate at the start of training |
| Reduce-on-Plateau | Automatically lowers the learning rate when validation performance stagnates |
| Main Goal | Improve convergence speed and final model performance |

> **Key Insight:** A fixed learning rate is rarely optimal throughout the entire optimization process. Learning rate scheduling improves training by allowing large parameter updates during the early stages of optimization and smaller, more precise updates near convergence. Modern deep learning commonly employs schedules such as cosine annealing, warm-up, and reduce-on-plateau to accelerate convergence, improve stability, and enhance generalization across a wide variety of neural network architectures.